In [1]:
!pip install -q transformers numpy accelerate datasets pandas evaluate

In [32]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from datasets import Dataset
from evaluate import load
from scipy.stats import pearsonr
from google.colab import drive


In [33]:
import os
from google.colab import drive

drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/ColabNotebooks'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['learn (1).ipynb', 'learn.ipynb', 'nlp_general_pipeline.ipynb', 'train_data.csv', 'test_data.csv', 'Untitled0.ipynb']


In [34]:
train_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_data.csv')
test_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/test_data.csv')

test_df["clean_word"] = (
    test_df["word"].fillna("")
    .str.replace("ţ", "ț")
    .str.replace("ş", "ș")
    .str.replace("Ţ", "Ț")
    .str.replace("Ş", "Ș")
)

In [35]:
MODEL_NAME = "readerbench/RoGPT2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

model.eval()


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: readerbench/RoGPT2-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [36]:
test_df["group_key"] = test_df["word_id"].astype(str).str.rsplit("_", n=1).str[0]
test_df["word_idx"] = test_df["word_id"].astype(str).str.rsplit("_", n=1).str[1].astype(int)

In [39]:
unique_words = (
    test_df[["group_key", "word_idx", "clean_word"]]
    .drop_duplicates()
    .sort_values(by=["group_key", "word_idx"])
)

print(f"total test words: {len(unique_words)}")


total test words: 1885


In [41]:
surprisal_map = {}

for group_key, group in unique_words.groupby("group_key"):
  words = group["clean_word"].tolist()
  indices = group["word_idx"].tolist()

  text, spans = "", []
  for w in words:
    spans.append((len(text), len(text) + len(w)))
    text += w + " "

  text = text.strip()

  enc = tokenizer(
      text,
      return_tensors = "pt",
      return_offsets_mapping = True,
      truncation = True,
      max_length = 1024
  )

  input_ids = enc["input_ids"].to(device)
  offsets = enc["offset_mapping"][0].tolist()

  with torch.no_grad():
    logits = model(input_ids = input_ids).logits[0]

    log_probs = torch.log_softmax(logits, dim = -1)

  # surprisal of token = -log P(token | previous token)

  tok_surp = [0.0] + [-log_probs[t - 1, input_ids[0, t]].item() for t in range(1, len(offsets))]

  for w_idx, (w_start, w_end) in zip(indices, spans):
    surprisal_map[(group_key, w_idx)] = sum(
        tok_surp[t] for t, (t_start, t_end) in enumerate(offsets)
        if t_end > w_start and t_start < w_end
    )






In [43]:
keys = list(zip(test_df["group_key"], test_df["word_idx"]))
preds = [surprisal_map.get(k, 0.0) for k in keys]

In [45]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["word_id"],
    "prediction": np.round(preds, 4)
})

OUTPUT_DIR = "/content/drive/MyDrive/ColabNotebooks/submission_rogpt2_zeroshot.csv"
submission.to_csv(OUTPUT_DIR, index = False)
print(submission.head(10))

   subtaskID                   datapointID  prediction
0          1  arg_pisacowsmilk_10_page_1_0      9.9051
1          1  arg_pisacowsmilk_10_page_1_1      1.6871
2          1  arg_pisacowsmilk_10_page_1_2      2.3801
3          1  arg_pisacowsmilk_10_page_1_3      9.4025
4          1  arg_pisacowsmilk_10_page_1_4      9.9793
5          1  arg_pisacowsmilk_10_page_1_5      6.2738
6          1  arg_pisacowsmilk_10_page_1_6      8.2201
7          1  arg_pisacowsmilk_10_page_1_7      6.6480
8          1  arg_pisacowsmilk_10_page_1_8      5.4916
9          1  arg_pisacowsmilk_10_page_1_9      6.5507
